In [ ]:
import re
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Sampler
from transformers import AutoModel, AutoTokenizer, get_cosine_schedule_with_warmup
import pandas as pd
import numpy as np
import faiss
import bitsandbytes as bnb
from tqdm.auto import tqdm
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from IPython.display import clear_output

In [ ]:
# ==========================================
# 1. ЗАГРУЗКА И ПРЕПРОЦЕССИНГ ДАННЫХ
# ==========================================
print("Загрузка данных...")
train_df = pd.read_parquet('train.parquet')
items_df = pd.read_parquet('benchmark_items.parquet')
queries_df = pd.read_parquet('benchmark_queries.parquet')

# Заполняем пропуски пустыми строками, чтобы не ломался конкатенатор текстов
train_df.fillna({'search_infm_params_text': '', 'item_infm_params_text': '', 'item_description_raw': ''}, inplace=True)
items_df.fillna({'item_infm_params_text': '', 'item_description_raw': ''}, inplace=True)
queries_df.fillna({'search_infm_params_text': ''}, inplace=True)

# --- Обработка категориальных признаков ---
# Я обучаю единый энкодер на всех данных, чтобы не было конфликтов индексов на инференсе
cat_cols = ['search_location_id', 'item_category_id', 'item_microcat_id']
encoders = {}

print("Обучение энкодеров...")
for col in cat_cols:
    le = LabelEncoder()
    all_values = pd.concat([
        train_df[col] if col in train_df else pd.Series([-1]*len(train_df)),
        items_df[col] if col in items_df else pd.Series([-1]*len(items_df)),
        queries_df[col] if col in queries_df else pd.Series([-1]*len(queries_df))
    ]).fillna(-1)

    le.fit(all_values)
    
    if col in train_df: train_df[f'{col}_idx'] = le.transform(train_df[col].fillna(-1))
    if col in items_df: items_df[f'{col}_idx'] = le.transform(items_df[col].fillna(-1))
    if col in queries_df: queries_df[f'{col}_idx'] = le.transform(queries_df[col].fillna(-1))
    encoders[col] = le

# --- Обработка числовых признаков ---
# Логарифмируем признаки с сильным разбросом
for df in [train_df, items_df]:
    if 'item_price' in df: df['price_log'] = np.log1p(df['item_price'].fillna(0).astype(float))
    if 'item_rating_reviews_count' in df: df['reviews_log'] = np.log1p(df['item_rating_reviews_count'].fillna(0).astype(float))
    if 'item_rating' in df: df['rating_clean'] = df['item_rating'].fillna(0).astype(float)

# У запросов этих данных нет, ставим заглушки
for col in ['price_log', 'reviews_log', 'rating_clean']:
    if col not in queries_df: queries_df[col] = 0.0

num_cols = ['price_log', 'reviews_log', 'rating_clean']
scaler = StandardScaler()

for df in [train_df, items_df, queries_df]:
    df[num_cols] = df[num_cols].replace([np.inf, -np.inf], np.nan).fillna(0.0)

train_df[num_cols] = scaler.fit_transform(train_df[num_cols])
items_df[num_cols] = scaler.transform(items_df[num_cols])
queries_df[num_cols] = scaler.transform(queries_df[num_cols])

In [ ]:
# ==========================================
# 2. КЛАССЫ ДАТАСЕТОВ И ТОКЕНИЗАТОР
# ==========================================
print("Инициализация токенизатора...")
tokenizer = AutoTokenizer.from_pretrained("BAAI/bge-m3")

# Получаем индексы неизвестных классов для маскирования
unknown_loc = encoders['search_location_id'].transform([-1])[0]
unknown_cat = encoders['item_category_id'].transform([-1])[0]
unknown_microcat = encoders['item_microcat_id'].transform([-1])[0]

def safe_str(val):
    s = str(val)
    return "" if s.lower() == 'nan' else s

class AvitoTrainDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # 1. Формируем текст запроса
        sq = safe_str(row.get('search_query', ''))
        sp = safe_str(row.get('search_infm_params_text', ''))
        q_text = f"{sq} {sp}".strip()

        # 2. Формируем структурированный текст объявления (описание обрезаю до 150 символов для скорости)
        title = safe_str(row.get('item_title_raw', ''))
        params = safe_str(row.get('item_infm_params_text', ''))
        desc = safe_str(row.get('item_description_raw', ''))

        parts = []
        if title: parts.append(f"название: {title}")
        if params: parts.append(f"характеристики: {params}")
        if desc: parts.append(f"описание: {desc[:150]}")
        pos_text = " | ".join(parts) if parts else "пусто"

        # ВАЖНО: Маскируем табличные фичи для запроса, чтобы модель не читерила
        q_cat = torch.tensor([unknown_loc, unknown_cat, unknown_microcat], dtype=torch.long)
        q_num = torch.tensor([0.0, 0.0, 0.0], dtype=torch.float32)

        # Для объявления отдаем реальные фичи
        pos_cat = torch.tensor([row['search_location_id_idx'], row['item_category_id_idx'], row['item_microcat_id_idx']], dtype=torch.long)
        pos_num = torch.tensor([row['price_log'], row['reviews_log'], row['rating_clean']], dtype=torch.float32)

        return q_text, pos_text, q_cat, q_num, pos_cat, pos_num

def collate_train(batch):
    q_texts, pos_texts, q_cats, q_nums, pos_cats, pos_nums = zip(*batch)
    q_tokens = tokenizer(list(q_texts), padding=True, truncation=True, max_length=64, return_tensors='pt')
    pos_tokens = tokenizer(list(pos_texts), padding=True, truncation=True, max_length=128, return_tensors='pt')
    
    return (
        q_tokens['input_ids'], q_tokens['attention_mask'], torch.stack(q_cats), torch.stack(q_nums),
        pos_tokens['input_ids'], pos_tokens['attention_mask'], torch.stack(pos_cats), torch.stack(pos_nums)
    )

class AvitoEvalDataset(Dataset):
    def __init__(self, df, is_query=False):
        self.df = df.reset_index(drop=True)
        self.is_query = is_query

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        obj_id = row['query_id'] if self.is_query and 'query_id' in row else row.get('item_id', 0)

        if self.is_query: 
            sq = safe_str(row.get('search_query', row.get('query', '')))
            sp = safe_str(row.get('search_infm_params_text', ''))
            text = f"{sq} {sp}".strip()
            cat_feat = torch.tensor([unknown_loc, unknown_cat, unknown_microcat], dtype=torch.long)
            num_feat = torch.tensor([0.0, 0.0, 0.0], dtype=torch.float32)
        else: 
            title = safe_str(row.get('item_title_raw', row.get('title', '')))
            params = safe_str(row.get('item_infm_params_text', row.get('params', '')))
            desc = safe_str(row.get('item_description_raw', row.get('description', '')))
            
            parts = []
            if title: parts.append(f"название: {title}")
            if params: parts.append(f"характеристики: {params}")
            if desc: parts.append(f"описание: {desc[:150]}")
            text = " | ".join(parts) if parts else "пусто"

            loc_idx = row.get('search_location_id_idx', unknown_loc)
            cat_idx = row.get('item_category_id_idx', unknown_cat)
            micro_idx = row.get('item_microcat_id_idx', unknown_microcat)
            cat_feat = torch.tensor([loc_idx, cat_idx, micro_idx], dtype=torch.long)
            num_feat = torch.tensor([row.get('price_log', 0.0), row.get('reviews_log', 0.0), row.get('rating_clean', 0.0)], dtype=torch.float32)

        return obj_id, text, cat_feat, num_feat

def collate_eval(batch):
    ids, texts, cat_feats, num_feats = zip(*batch)
    tokens = tokenizer(list(texts), padding=True, truncation=True, max_length=128, return_tensors='pt')
    return list(ids), tokens['input_ids'], tokens['attention_mask'], torch.stack(cat_feats), torch.stack(num_feats)

In [ ]:
# ==========================================
# 3. ПОДГОТОВКА ДАТАЛОАДЕРОВ (HARD NEGATIVES)
# ==========================================
# Мой кастомный сэмплер для хард-негативов. Он берет отсортированный по категориям датасет,
# режет его на куски по batch_size, и выдает эти куски в случайном порядке каждую эпоху.
class ChunkedSampler(Sampler):
    def __init__(self, data_source, batch_size):
        self.data_source = data_source
        self.batch_size = batch_size
        self.chunks = []
        for i in range(0, len(data_source), batch_size):
            self.chunks.append(list(range(i, min(i + batch_size, len(data_source)))))

    def __iter__(self):
        random.shuffle(self.chunks) 
        flattened_indices = [idx for chunk in self.chunks for idx in chunk]
        return iter(flattened_indices)

    def __len__(self):
        return len(self.data_source)

print("Разбиение на train/val...")
# Честная валидация: отрезаем 5% ДО сортировки по категориям
train_df, val_df = train_test_split(train_df, test_size=0.05, random_state=42)

print("Сортировка трейна для ChunkedSampler...")
train_df = train_df.sort_values(by='item_category_id').reset_index(drop=True)

train_dataset = AvitoTrainDataset(train_df)
chunked_sampler = ChunkedSampler(train_dataset, batch_size=128)

train_loader = DataLoader(
    train_dataset, 
    batch_size=128, 
    sampler=chunked_sampler, # <- Формирует батчи внутри одной категории
    collate_fn=collate_train, 
    num_workers=0
)

val_queries_loader = DataLoader(AvitoEvalDataset(val_df, is_query=True), batch_size=128, collate_fn=collate_eval, num_workers=0)
val_items_loader = DataLoader(AvitoEvalDataset(val_df, is_query=False), batch_size=128, collate_fn=collate_eval, num_workers=0)

In [ ]:
# ==========================================
# 4. АРХИТЕКТУРА И ЛОСС
# ==========================================
class AvitoHybridEncoder(nn.Module):
    def __init__(self, model_name="BAAI/bge-m3", cat_vocab_sizes=[1000, 500, 5000], cat_emb_dim=32, num_features_dim=3, final_out_dim=768):
        super().__init__()
        # Текстовый энкодер
        self.text_model = AutoModel.from_pretrained(model_name)
        text_hidden_size = self.text_model.config.hidden_size
        
        # Эмбеддинги для категорий и локаций
        self.cat_embeddings = nn.ModuleList([nn.Embedding(num_classes, cat_emb_dim) for num_classes in cat_vocab_sizes])
        
        # MLP-блок для слияния признаков
        fusion_dim = text_hidden_size + (len(cat_vocab_sizes) * cat_emb_dim) + num_features_dim
        self.mlp = nn.Sequential(nn.Linear(fusion_dim, 1024), nn.GELU(), nn.LayerNorm(1024), nn.Linear(1024, final_out_dim))

    def forward(self, input_ids, attention_mask, cat_features, num_features):
        text_out = self.text_model(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state[:, 0, :] 
        cat_embs = [emb_layer(cat_features[:, i]) for i, emb_layer in enumerate(self.cat_embeddings)]
        cat_out = torch.cat(cat_embs, dim=1)
        combined_features = torch.cat([text_out, cat_out, num_features], dim=1)
        
        final_embedding = self.mlp(combined_features)
        return torch.nn.functional.normalize(final_embedding, p=2, dim=1)

# Использую InfoNCE Loss. Температура 0.05 заставляет модель более жестко штрафовать хард-негативы.
def compute_infonce_loss(q_emb, pos_emb, temperature=0.05):
    logits = torch.matmul(q_emb, pos_emb.T) / temperature
    labels = torch.arange(logits.size(0), device=logits.device)
    return F.cross_entropy(logits, labels)

In [ ]:
# ==========================================
# 5. ФУНКЦИИ ВАЛИДАЦИИ И ИНФЕРЕНСА
# ==========================================
def evaluate_recall_at_k(model, v_q_loader, v_i_loader, device='cuda', k=50):
    model.eval()
    item_embeddings, item_ids_mapping = [], []
    
    with torch.no_grad():
        for batch in tqdm(v_i_loader, desc="Векторизация базы валидации"):
            ids, input_ids, mask, cat_feat, num_feat = [x.to(device) if isinstance(x, torch.Tensor) else x for x in batch]
            with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
                emb = model(input_ids, mask, cat_feat, num_feat)
            item_embeddings.append(emb.float().cpu().numpy())
            item_ids_mapping.extend(ids)

    item_embeddings_matrix = np.vstack(item_embeddings)
    index = faiss.IndexFlatIP(item_embeddings_matrix.shape[1]) 
    index.add(item_embeddings_matrix)

    hits, total_queries = 0, 0
    with torch.no_grad():
        for batch in tqdm(v_q_loader, desc="Подсчет Recall"):
            target_ids, input_ids, mask, cat_feat, num_feat = [x.to(device) if isinstance(x, torch.Tensor) else x for x in batch]
            with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
                q_emb = model(input_ids, mask, cat_feat, num_feat)
            distances, faiss_indices = index.search(q_emb.float().cpu().numpy(), k)

            for i in range(len(target_ids)):
                if target_ids[i] in [item_ids_mapping[idx] for idx in faiss_indices[i]]: hits += 1
                total_queries += 1

    recall = hits / total_queries
    print(f"✅ Recall@{k}: {recall:.4f}")
    return recall

def generate_submission(model, b_items_loader, b_queries_loader, device='cuda', k=50):
    model.eval()
    item_embeddings, item_ids_mapping = [], []
    with torch.no_grad():
        for batch in tqdm(b_items_loader, desc="1/3 Векторизация benchmark_items"):
            ids, input_ids, mask, cat_feat, num_feat = [x.to(device) if isinstance(x, torch.Tensor) else x for x in batch]
            with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
                emb = model(input_ids, mask, cat_feat, num_feat)
            item_embeddings.append(emb.float().cpu().numpy())
            item_ids_mapping.extend(ids)

    print("2/3 Сборка индекса FAISS...")
    item_embeddings_matrix = np.vstack(item_embeddings)
    index = faiss.IndexFlatIP(item_embeddings_matrix.shape[1])
    index.add(item_embeddings_matrix)

    submission_data = []
    with torch.no_grad():
        for batch in tqdm(b_queries_loader, desc="3/3 Поиск топ-50 кандидатов"):
            query_ids, input_ids, mask, cat_feat, num_feat = [x.to(device) if isinstance(x, torch.Tensor) else x for x in batch]
            with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
                q_emb = model(input_ids, mask, cat_feat, num_feat)

            distances, faiss_indices = index.search(q_emb.float().cpu().numpy(), k)
            for i, q_id in enumerate(query_ids):
                submission_data.append({
                    'query_id': q_id,
                    'predicted_item_ids': [item_ids_mapping[idx] for idx in faiss_indices[i]]
                })
    return pd.DataFrame(submission_data)

In [ ]:
# ==========================================
# 6. ОБУЧЕНИЕ (РАЗМОРОЗКА + ЧЕКПОИНТИНГ)
# ==========================================
device = 'cuda' 
model = AvitoHybridEncoder(
    model_name="BAAI/bge-m3", 
    cat_vocab_sizes=[len(encoders['search_location_id'].classes_), len(encoders['item_category_id'].classes_), len(encoders['item_microcat_id'].classes_)]
).to(device)

# Включаю Gradient Checkpointing, чтобы обучать все слои трансформера без ошибки Out of Memory
model.text_model.gradient_checkpointing_enable()

# Полная разморозка модели для лучшей адаптации под данные Авито
for param in model.parameters():
    param.requires_grad = True

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Обучаемых параметров: {trainable_params:,}")

# Использую 8-битный оптимизатор для экономии памяти видеокарты
optimizer = bnb.optim.AdamW8bit(model.parameters(), lr=2e-5, weight_decay=0.01)
scaler = torch.cuda.amp.GradScaler()

epochs = 4
accumulation_steps = 2
best_recall = 0.0

total_steps = (len(train_loader) // accumulation_steps) * epochs
scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=int(total_steps * 0.1), num_training_steps=total_steps)

for epoch in range(epochs):
    model.train()
    total_loss = 0
    optimizer.zero_grad()
    
    progress_bar = tqdm(train_loader, desc=f"Эпоха {epoch+1}/{epochs}")
    for i, batch in enumerate(progress_bar):
        q_ids, q_mask, q_cat, q_num, pos_ids, pos_mask, pos_cat, pos_num = [x.to(device) if isinstance(x, torch.Tensor) else x for x in batch]

        with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
            loss = compute_infonce_loss(model(q_ids, q_mask, q_cat, q_num), model(pos_ids, pos_mask, pos_cat, pos_num)) / accumulation_steps

        scaler.scale(loss).backward()

        if (i + 1) % accumulation_steps == 0:
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()

        total_loss += loss.item() * accumulation_steps
        progress_bar.set_postfix({'loss': total_loss / (i + 1)})

    print("\nЗапуск валидации...")
    current_recall = evaluate_recall_at_k(model, val_queries_loader, val_items_loader, device=device, k=50)

    if current_recall > best_recall:
        best_recall = current_recall
        torch.save(model.state_dict(), "best_avito_hybrid_model.pth")
        print(f"🌟 Новая лучшая модель сохранена! Recall@50: {best_recall:.4f}")

In [ ]:
# ==========================================
# 7. ГЕНЕРАЦИЯ ФИНАЛЬНОГО РЕШЕНИЯ
# ==========================================
print("\nЗагрузка лучших весов для инференса...")
model.load_state_dict(torch.load("best_avito_hybrid_model.pth"))

# Подготовка лоадеров для генерации итогового файла
benchmark_items_loader = DataLoader(AvitoEvalDataset(items_df, is_query=False), batch_size=128, collate_fn=collate_eval, num_workers=0)
benchmark_queries_loader = DataLoader(AvitoEvalDataset(queries_df, is_query=True), batch_size=128, collate_fn=collate_eval, num_workers=0)

submission_df = generate_submission(model, benchmark_items_loader, benchmark_queries_loader, device=device)

# Склеиваем списки в строку через пробел по формату Авито
submission_df['answer'] = submission_df['predicted_item_ids'].apply(lambda x: ' '.join(map(str, x)))
final_answer = submission_df[['query_id', 'answer']]

final_answer.to_csv('answer_final.csv', index=False)
print("Готово! Файл answer_final.csv собран и готов к отправке.")